# 03 - Optimization Model
Use stochastic delay scenarios and robust MILP constraints to build a safer procurement plan.

## Step 1: Prepare inputs
Load data, retrain delay model, and append predicted delays.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Ensure project root (folder containing src/) is importable in notebook sessions.
project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_generation import default_component_demand
from src.delay_predictor import train_delay_model, predict_delays
from src.optimizer import generate_delay_scenarios, optimize_procurement

data_path = project_root / "data" / "supply_chain_dataset.csv"
df = pd.read_csv(data_path)
artifacts = train_delay_model(df)
df['predicted_delay_days'] = predict_delays(artifacts.pipeline, df)

## Step 2: Run optimization
Build demand, generate scenarios, and solve the robust MILP.

In [ ]:
demand = default_component_demand(df)
scenarios = generate_delay_scenarios(
    predicted_delay_days=df['predicted_delay_days'].to_numpy(),
    residual_std=artifacts.metrics['residual_std'],
    n_scenarios=80,
    seed=42,
)
plan, summary = optimize_procurement(
    df=df,
    demand=demand,
    budget=2_000_000,
    deadline_days=28,
    delay_penalty=4.0,
    delay_scenarios=scenarios,
    risk_weight=0.2,
    scenario_confidence=0.9,
    delay_variability_cap=3.0,
    solver_time_limit=10,
)
summary

## Step 3: Preview plan
Show the top selected procurement rows.

In [ ]:
plan.head()